# Task 1 – Uncertainty Scores
**Owner:** Sai Akhil Singade · DSA1 · Action Learning

## 1. Introduction
**Research context.** #1 Vashurin et al. 2025 (LM-Polygraph benchmark, PRR), #2 Farquhar et al. 2024
(semantic entropy), #3 Kossen et al. 2024 (hidden-state probes and their out-of-distribution behaviour),
#4 Shelmanov et al. 2025 (hidden-state detectors may memorise topics).

**Objectives.** Compare three uncertainty scores for detecting wrong answers of Qwen3-1.7B/4B/8B, in domain
(TriviaQA) and under shift (BioASQ, MMLU medical), and measure their compute cost.

**Research question.** Which score best ranks wrong answers above right ones on TriviaQA, and how much of
that ranking survives on BioASQ and MMLU medical?

**Hypothesis.** The probe is strongest in domain but loses most under shift; semantic entropy transfers best;
token probability is cheapest and in between.

**Convention.** Every score is an *uncertainty*: higher = more likely wrong. Positive class = wrong answer.

## 2. Setup

In [ ]:
# --- Setup (implementation guide, section 2). Run once per session. ------------------------------
import os, sys, subprocess, random, json
from pathlib import Path

REPO_URL = "https://github.com/adiarora28/dsa1-conformal-abstention.git"   # <- set once for the team
ON_KAGGLE = Path("/kaggle").exists()
REPO_DIR = Path("/kaggle/working/dsa1-conformal-abstention") if ON_KAGGLE else Path.cwd().resolve().parent
if ON_KAGGLE and not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
if ON_KAGGLE:  # upgrade only what Kaggle's image may lack; never reinstall torch
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers>=4.56",
                    "accelerate", "bitsandbytes", "sentence-transformers", "xgboost"], check=True)
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
          "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
except ImportError:
    print("PyTorch not installed (this notebook can run on CPU)")

from dsa1 import config as C
from dsa1.cache_io import link_cache_roots
CACHE_ROOT = link_cache_roots("/kaggle/input", "/kaggle/working/cache_merged") if ON_KAGGLE else Path("cache")
print("CACHE_ROOT:", CACHE_ROOT)
if ON_KAGGLE:  # bring outputs of earlier notebooks (attached as inputs) into this working copy
    import shutil
    Path("results").mkdir(exist_ok=True)
    for p in Path("/kaggle/input").rglob("semantic_entropy"):
        if p.is_dir():
            shutil.copytree(p, "results/semantic_entropy", dirs_exist_ok=True)
    for name in ["hand_check.csv", "failure_cases.csv"]:
        hits = sorted(Path("/kaggle/input").rglob(name))
        if hits and not (Path("results") / name).exists():
            shutil.copy(hits[0], Path("results") / name)
            print("copied", hits[0])
for p in sorted(Path(CACHE_ROOT).glob("*/*")):
    print("  ", p.relative_to(CACHE_ROOT))


In [ ]:
from dsa1.splits import load_splits
SPLITS = load_splits("data/splits/splits.json")
SEEDS = [int(s) for s in SPLITS["seeds"]]
TEST_KEYS = {"triviaqa_pool": "test_triviaqa", "bioasq": "test_bioasq", "mmlu_med": "test_mmlu_med"}
print("seeds:", SEEDS, "| sizes (seed 0):", {k: len(v) for k, v in SPLITS["seeds"]["0"].items()})


## 3. Data loading & exploration

In [ ]:
from dsa1 import experiments as E
from dsa1.cache_io import cache_dir, load_records
MODELS = list(C.MODELS)
DATASETS = ["triviaqa_pool", "bioasq", "mmlu_med"]
base, rows = {}, []
for m in MODELS:
    for ds in DATASETS:
        try:
            df = E.base_scores(CACHE_ROOT, m, ds)
        except FileNotFoundError:
            print("missing:", m, ds)
            continue
        base[(m, ds)] = df
        rows.append({"model": m, "dataset": ds, "n": len(df), "accuracy (EM)": 1 - df["err"].mean(),
                     "accuracy (F1>=0.5)": 1 - df["err_f1"].mean(), "semantic entropy ready": df["u_se"].notna().mean()})
accuracy = pd.DataFrame(rows)
accuracy.to_csv("results/metrics/task1_accuracy.csv", index=False)
accuracy

In [ ]:
# Five examples per open-ended domain (Qwen3-4B), including wrong ones
for ds in ["triviaqa_pool", "bioasq"]:
    rec = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", ds))
    display(rec.sample(5, random_state=0)[["question", "answer", "aliases", "correct_em", "f1"]])

In [ ]:
# Hand-check agreement (Step 0.5): fill results/hand_check.csv, column human_correct = 1/0
hc = Path("results/hand_check.csv")
if hc.exists():
    h = pd.read_csv(hc)
    h = h[h["human_correct"].astype(str).str.strip().isin(["0", "1"])]
    if len(h):
        agree = (h["human_correct"].astype(int) == h["correct_em"].astype(int)).mean()
        print(f"Exact-match labels agree with manual labels on {agree:.1%} of {len(h)} answers")
        display(h[h["human_correct"].astype(int) != h["correct_em"].astype(int)].head(10))

## 4. Preprocessing
Splits come from `data/splits/splits.json` (five seeds). Answers are normalised SQuAD-style before exact match.
The probe trains only on the fixed 2,000-question TriviaQA probe set, which is disjoint from every
calibration and test set.

In [ ]:
for (m, ds), df in base.items():
    for s in SEEDS:
        missing = set(SPLITS["seeds"][str(s)][TEST_KEYS[ds]]) - set(df["qid"])
        assert not missing, f"{m}/{ds} seed {s}: {len(missing)} test questions missing from cache"
print("all test questions are in the cache")

## 5. Model definition
- Qwen3-1.7B and 4B in float16, Qwen3-8B in 4-bit NF4; thinking disabled; greedy answer for all, plus 10 samples (T=1.0, top-p 0.95) for 1.7B and 4B.
- **V1 token probability:** minus the mean token log-probability of the greedy answer (MMLU: minus log-prob of the chosen option).
- **V2 semantic entropy:** 10 samples clustered by bidirectional entailment with DeBERTa-large-MNLI (computed on GPU by
  `scripts/compute_semantic_entropy.py`); MMLU uses the entropy of the four option probabilities.
- **V3 probe:** logistic regression on one hidden state; position (last prompt / last answer token), layer
  (25/50/100% depth) and C chosen by 5-fold CV on the probe set only.

In [ ]:
import joblib
probes = {}
for m in MODELS:
    try:
        probe, pos, layer, table = E.select_probe(CACHE_ROOT, m)
    except FileNotFoundError:
        print("no probe set for", m)
        continue
    probes[m] = (probe, pos, layer)
    joblib.dump(probe, f"results/probe_{m}.joblib")
    print(m, "-> best:", pos, "token, layer slot", layer)
    display(table.assign(model=m))

## 6. Running the variations (apply the probe, write the score tables used by Tasks 2 and 3)

In [ ]:
for (m, ds), df in list(base.items()):
    if m in probes:
        probe, pos, layer = probes[m]
        df = E.add_probe_scores(df, probe, CACHE_ROOT, m, ds, pos, layer)
    else:
        df = df.assign(u_probe=np.nan)
    base[(m, ds)] = df
    E.save_score_table(df, m, ds)
print("score tables written to results/scores/<model>/<dataset>.parquet")

Optional cross-check with LM-Polygraph (guide Task 1, Steps 2–3): run its maximum-sequence-probability and
semantic-entropy estimators on 50 TriviaQA questions with Qwen3-4B, save `results/polygraph_check.csv` with
columns `qid, polygraph_msp, polygraph_se`, then run the next cell. If LM-Polygraph does not run with Qwen3,
record it as a technical challenge.

In [ ]:
pc = Path("results/polygraph_check.csv")
if pc.exists():
    from scipy.stats import spearmanr
    chk = pd.read_csv(pc).merge(base[("qwen3-4b", "triviaqa_pool")], on="qid")
    print("Spearman token prob vs LM-Polygraph MSP:", spearmanr(chk["u_tokprob"], -chk["polygraph_msp"]).statistic)
    print("Spearman semantic entropy vs LM-Polygraph:", spearmanr(chk["u_se"], chk["polygraph_se"]).statistic)

## 7. Evaluation – AUROC and PRR, mean ± std over five seeds

In [ ]:
from dsa1.metrics import auroc_wrong, prediction_rejection_ratio, paired_bootstrap_auroc_diff, mean_std
SCORES = ["u_tokprob", "u_se", "u_probe"]
rows = []
for (m, ds), df in base.items():
    for s in SEEDS:
        sub = E.rows_for(df, SPLITS["seeds"][str(s)][TEST_KEYS[ds]])
        for sc in SCORES:
            if sub[sc].isna().any():
                continue
            rows.append({"model": m, "dataset": ds, "seed": s, "score": sc,
                         "auroc": auroc_wrong(sub[sc], sub["err"]),
                         "prr": prediction_rejection_ratio(sub[sc], sub["err"])})
res = pd.DataFrame(rows)
res.to_csv("results/metrics/task1_auroc_prr.csv", index=False)
for metric in ["auroc", "prr"]:
    print(metric.upper(), "(MMLU uses all 945 questions in every seed, so its std is 0)")
    display(res.pivot_table(index="score", columns=["model", "dataset"], values=metric, aggfunc=mean_std))

In [ ]:
# Is the difference between two scores real? Paired bootstrap on seed 0, Qwen3-4B
rows = []
for ds in DATASETS:
    if ("qwen3-4b", ds) not in base:
        continue
    sub = E.rows_for(base[("qwen3-4b", ds)], SPLITS["seeds"]["0"][TEST_KEYS[ds]])
    for a, b in [("u_se", "u_tokprob"), ("u_probe", "u_tokprob"), ("u_probe", "u_se")]:
        if sub[[a, b]].isna().any().any():
            continue
        rows.append({"dataset": ds, "comparison": f"{a} - {b}", **paired_bootstrap_auroc_diff(sub["err"], sub[a], sub[b])})
pd.DataFrame(rows)

In [ ]:
# Cost per score (GPU seconds per question, from the cache timings)
rows = []
for (m, ds), df in base.items():
    for sc in SCORES:
        rows.append({"model": m, "dataset": ds, "score": sc, "gpu_s_per_question": E.cost_per_question(df, sc).mean(),
                     "extra_generations": 10 if sc.startswith("u_se") and ds != "mmlu_med" else 0})
cost = pd.DataFrame(rows)
cost.to_csv("results/metrics/task1_cost.csv", index=False)
cost.pivot_table(index="score", columns=["model", "dataset"], values="gpu_s_per_question").round(3)

## 8. Comparison & analysis

In [ ]:
# Grouped bars: AUROC by score and domain, one panel per model
fig, axes = plt.subplots(1, len(MODELS), figsize=(5 * len(MODELS), 4), sharey=True)
for ax, m in zip(np.atleast_1d(axes), MODELS):
    g = res[res["model"] == m].groupby(["dataset", "score"])["auroc"].agg(["mean", "std"]).unstack()
    if g.empty:
        continue
    g["mean"].plot.bar(yerr=g["std"], ax=ax, rot=0, capsize=3)
    ax.set_title(m); ax.set_ylabel("AUROC (wrong answer)"); ax.set_ylim(0.4, 1.0); ax.axhline(0.5, ls=":", c="grey")
plt.tight_layout(); plt.savefig("results/figures/task1_auroc_bars.png", dpi=150); plt.show()

In [ ]:
# ROC curves, Qwen3-4B, seed 0
from sklearn.metrics import roc_curve
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, ds in zip(axes, DATASETS):
    if ("qwen3-4b", ds) not in base:
        continue
    sub = E.rows_for(base[("qwen3-4b", ds)], SPLITS["seeds"]["0"][TEST_KEYS[ds]])
    for sc in SCORES:
        if sub[sc].isna().any():
            continue
        fpr, tpr, _ = roc_curve(sub["err"], sub[sc])
        ax.plot(fpr, tpr, label=f"{sc} ({auroc_wrong(sub[sc], sub['err']):.2f})")
    ax.plot([0, 1], [0, 1], ":", c="grey"); ax.set_title(f"Qwen3-4B – {ds}")
    ax.set_xlabel("false positive rate"); ax.set_ylabel("true positive rate (wrong answers caught)"); ax.legend()
plt.tight_layout(); plt.savefig("results/figures/task1_roc_4b.png", dpi=150); plt.show()

In [ ]:
# Deeper analysis 1 – transfer gap: in-domain AUROC minus shifted AUROC (positive = worse under shift)
piv = res.pivot_table(index=["model", "score", "seed"], columns="dataset", values="auroc")
gap = pd.DataFrame({"to BioASQ": piv["triviaqa_pool"] - piv.get("bioasq"),
                    "to MMLU med": piv["triviaqa_pool"] - piv.get("mmlu_med")}).reset_index()
gsum = gap.groupby(["model", "score"])[["to BioASQ", "to MMLU med"]].agg(["mean", "std"])
display(gsum.round(3))
fig, ax = plt.subplots(figsize=(8, 4))
for i, (key, row) in enumerate(gsum.iterrows()):
    ax.errorbar(i, row[("to BioASQ", "mean")], yerr=row[("to BioASQ", "std")], fmt="o", capsize=3)
ax.set_xticks(range(len(gsum))); ax.set_xticklabels([f"{m}\n{s}" for m, s in gsum.index], rotation=60, fontsize=8)
ax.axhline(0, c="grey", ls=":"); ax.set_ylabel("AUROC drop TriviaQA → BioASQ"); ax.set_title("Transfer gap per score and model")
plt.tight_layout(); plt.savefig("results/figures/task1_transfer_gap.png", dpi=150); plt.show()

In [ ]:
# AUROC versus compute cost (efficiency vs performance, as in the implementation guide)
merged = res.groupby(["model", "dataset", "score"])["auroc"].mean().reset_index().merge(cost, on=["model", "dataset", "score"])
fig, ax = plt.subplots(figsize=(7, 4))
for sc, g in merged[merged["dataset"] == "bioasq"].groupby("score"):
    ax.scatter(g["gpu_s_per_question"], g["auroc"], label=sc, s=60)
    for _, r in g.iterrows():
        ax.annotate(r["model"].replace("qwen3-", ""), (r["gpu_s_per_question"], r["auroc"]), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("GPU seconds per question (log)"); ax.set_ylabel("AUROC on BioASQ"); ax.legend()
ax.set_title("Quality vs cost under shift"); plt.tight_layout(); plt.savefig("results/figures/task1_cost_quality.png", dpi=150); plt.show()

In [ ]:
# Deeper analysis 2 – failure cases (Qwen3-4B, seed 0): export a sheet to label by hand
rows = []
for ds in ["triviaqa_pool", "bioasq"]:
    sub = E.rows_for(base[("qwen3-4b", ds)], SPLITS["seeds"]["0"][TEST_KEYS[ds]])
    rec = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", ds)).set_index("qid")
    for sc in SCORES:
        if sub[sc].isna().any():
            continue
        lo, hi = sub[sc].quantile(0.2), sub[sc].quantile(0.8)
        cw = sub[(sub[sc] <= lo) & (sub["err"] == 1)]
        ur = sub[(sub[sc] >= hi) & (sub["err"] == 0)]
        for kind, part in [("confident_but_wrong", cw), ("uncertain_but_right", ur)]:
            for q in part.sample(min(50, len(part)), random_state=0)["qid"]:
                rows.append({"dataset": ds, "score": sc, "kind": kind, "qid": q, "question": rec.loc[q, "question"],
                             "answer": rec.loc[q, "answer"], "aliases": " | ".join(list(rec.loc[q, "aliases"])[:8]), "category": ""})
fc = pd.DataFrame(rows)
fc_path = Path("results/failure_cases.csv")
if fc_path.exists() and pd.read_csv(fc_path)["category"].notna().any():
    print("results/failure_cases.csv already has labels - not overwritten")
else:
    fc.to_csv(fc_path, index=False)
print("Label column 'category' with: grading/alias error, partly correct, entity confusion, ambiguous question, knowledge gap")
fc.groupby(["dataset", "score", "kind"]).size().unstack()

In [ ]:
# After labelling: category counts, and how often the scores fail on the same questions
fc = pd.read_csv("results/failure_cases.csv")
if fc["category"].notna().any():
    display(fc.dropna(subset=["category"]).pivot_table(index="category", columns=["dataset", "score", "kind"],
                                                         values="qid", aggfunc="count", fill_value=0))
for ds in ["triviaqa_pool", "bioasq"]:
    sets = {sc: set(g["qid"]) for sc, g in fc[(fc["dataset"] == ds) & (fc["kind"] == "confident_but_wrong")].groupby("score")}
    keys = list(sets)
    for i in range(len(keys)):
        for j in range(i + 1, len(keys)):
            a, b = sets[keys[i]], sets[keys[j]]
            print(f"{ds}: overlap of confident-but-wrong sets {keys[i]} vs {keys[j]}: {len(a & b) / max(1, len(a | b)):.0%}")

### Confusion matrix and demo (for the defense)

In [ ]:
# Confusion matrix (defense guide): flag as many BioASQ answers as there are wrong ones, by each score
sub = E.rows_for(base[("qwen3-4b", "bioasq")], SPLITS["seeds"]["0"]["test_bioasq"])
panels, rows = [], []
for sc in SCORES:
    if sub[sc].isna().any():
        continue
    c = E.detection_confusion(sub[sc], sub["err"])
    panels.append((sc, c)); rows.append({"score": sc, **c})
pd.DataFrame(rows).to_csv("results/metrics/task1_confusion.csv", index=False)
E.plot_confusions(panels, "detection", "results/figures/task1_confusion.png", "Qwen3-4B · BioASQ · seed 0")
plt.show()
pd.DataFrame(rows)

In [ ]:
# Demo – inference on test samples: the model's own answers to three BioASQ test questions, with all scores
pd.set_option("display.max_colwidth", 80)
rec = load_records(cache_dir(CACHE_ROOT, "qwen3-4b", "bioasq")).set_index("qid")
lo, hi = sub["u_tokprob"].quantile(0.2), sub["u_tokprob"].quantile(0.8)
cases = {"confident and right": sub[(sub["u_tokprob"] <= lo) & (sub["err"] == 0)],
         "uncertain and wrong (caught)": sub[(sub["u_tokprob"] >= hi) & (sub["err"] == 1)],
         "confident but wrong (missed)": sub[(sub["u_tokprob"] <= lo) & (sub["err"] == 1)]}
rows = []
for case, part in cases.items():
    if len(part) == 0:
        continue
    q = part.sample(1, random_state=1).iloc[0]
    r = rec.loc[q["qid"]]
    rows.append({"case": case, "question": r["question"], "model answer": r["answer"],
                 "reference": " | ".join(list(r["aliases"])[:3]), "correct": not bool(q["err"]),
                 **{sc: round(float(q[sc]), 3) for sc in SCORES if not pd.isna(q[sc])}})
demo = pd.DataFrame(rows)
demo.to_csv("results/metrics/task1_demo_examples.csv", index=False)
demo

## 9. Discussion
*Write after the results are in. Answer each point in 2–3 sentences.*
- Which score is best in domain, and is the difference significant?
- Which score transfers best to BioASQ and MMLU medical? Does the probe's in-domain advantage survive?
- What do the failure categories say: score failures or model failures (shared confident errors)?
- Cost: is semantic entropy's quality worth 10 extra generations?
- Limitations: exact-match labels (quote the hand-check agreement), single model family, no semantic entropy for Qwen3-8B (4-bit sampling exceeded the free GPU budget).
- Link to papers #1–#4, and the MVP implication: which score the assistant should use.

## 10. References

- Bibliography numbers refer to `DSA 1 - Bibliography.xlsx`.
- Datasets: TriviaQA (Joshi et al., 2017), BioASQ Task B 14b (Krithara et al., 2023), MMLU (Hendrycks et al., 2021).
- Models: Qwen3 (Qwen Team, 2025); DeBERTa-large-MNLI (He et al., 2021); all-MiniLM-L6-v2 (Reimers & Gurevych, 2019).
- Code: shared `dsa1` package in this repository; LM-Polygraph (Vashurin et al., 2025) used for cross-checks;
  DS-CP reference code https://github.com/zhexiaolin/CP (Lin et al., 2026).
